In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Innlastning av dataset #

In [11]:
df_links_small = pd.read_csv('data/links_small.csv')
df_links_small.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [14]:
df_links = pd.read_csv('data/links.csv')
df_links.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [12]:
df_ratings_small = pd.read_csv('data/ratings_small.csv')
df_ratings_small.head()

,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179
2,1,1061,3.0,1260759182
3,1,1129,2.0,1260759185
4,1,1172,4.0,1260759205


In [13]:
df_ratings = pd.read_csv('data/ratings.csv')
df_ratings.head()

,userId,movieId,rating,timestamp
0,1,110,1.0,1425941529
1,1,147,4.5,1425942435
2,1,858,5.0,1425941523
3,1,1221,5.0,1425941546
4,1,1246,5.0,1425941556


-------------------------------------------------

# Utforskning av data #
Utforsker de ulike csv-filene ved hjelp av .shape, .head(), .dtypes og .info()

## Links_small ##

### .shape ###

In [ ]:
df_links_small.shape

### .head() ###

In [ ]:
df_links_small.head()

### .dtypes ###

In [ ]:
df_links_small.dtypes

### .info() ###

In [ ]:
df_links_small.info()

## Links ##

### .shape ###

In [ ]:
df_links.shape

### .head() ###

In [ ]:
df_links.head()

### .dtypes ###

In [ ]:
df_links.dtypes

### .info() ###

In [ ]:
df_links.info()

## Ratings_small ##

### .shape ###

In [ ]:
df_ratings_small.shape

### .head() ###

In [ ]:
df_ratings_small.head()

### .dtypes ###

In [ ]:
df_ratings_small.dtypes

### .info() ###

In [ ]:
df_ratings_small.info()

## Ratings ##

### .shape ###

In [ ]:
df_ratings.shape

### .head() ###

In [ ]:
df_ratings.head()

### .dtypes ###

In [ ]:
df_ratings.dtypes

### .info() ###

In [ ]:
df_ratings.info()

-------------------------------------------------

# Videre analyse av links og ratings #

## Links og links_small ##

### 1. Manglende verdier

In [ ]:
print("links_small:\n", df_links_small.isna().sum(), "\n")
print("links:\n", df_links.isna().sum())

Bare `tmdbId` mangler verdier: 13 i links_small og 219 i links. Det er også grunnen til at kolonnen er `float64` (NaN tvinger heltall over til float). Filmer uten `tmdbId` kan ikke kobles til `movies_metadata`, `credits` eller `keywords`, og vil derfor ikke ha metadata etter innsetting.

### 2. Duplikater

In [ ]:
for navn, df in [("links_small", df_links_small), ("links", df_links)]:
    print(navn)
    print("  Duplikate movieId:", df["movieId"].duplicated().sum())
    print("  Duplikate imdbId: ", df["imdbId"].duplicated().sum())
    print("  Duplikate tmdbId: ", df["tmdbId"].dropna().duplicated().sum())

In [ ]:
# Ser på noen av radene der flere movieId peker på samme tmdbId
tmdb_dup = df_links[df_links["tmdbId"].notna() & df_links["tmdbId"].duplicated(keep=False)]
tmdb_dup.sort_values("tmdbId").head(6)

`movieId` og `imdbId` er unike. I links peker 30 `tmdbId` på mer enn én `movieId`, altså to MovieLens-filmer som kobles til samme TMDB-film. Det betyr at koblingen links → movies_metadata ikke er helt én-til-én, og ratings for én TMDB-film kan være spredt på flere `movieId`. Dette må håndteres ved innsetting (f.eks. velge én eller slå sammen).

### 3. Er links_small et delsett av links?

In [ ]:
mangler_i_links = ~df_links_small["movieId"].isin(df_links["movieId"])
print("movieId i links_small som ikke finnes i links:", mangler_i_links.sum())
df_links_small[mangler_i_links]

Nesten, men ikke helt: 10 filmer i links_small finnes ikke i links. Small-versjonen er altså ikke et rent utsnitt av den fulle.

## Ratings og ratings_small ##

### 1. Manglende verdier og duplikater

In [ ]:
for navn, df in [("ratings_small", df_ratings_small), ("ratings", df_ratings)]:
    print(navn)
    print("  Manglende verdier totalt:  ", df.isna().sum().sum())
    print("  Helt identiske rader:      ", df.duplicated().sum())
    print("  Duplikate (userId, movieId):", df.duplicated(subset=["userId", "movieId"]).sum())

Ingen manglende verdier og ingen duplikater. Hver bruker har vurdert hver film maks én gang, så `(userId, movieId)` fungerer som en naturlig nøkkel.

### 2. Gyldige ratingverdier

In [ ]:
print("ratings_small:", sorted(df_ratings_small["rating"].unique().tolist()))
print("ratings:      ", sorted(df_ratings["rating"].unique().tolist()))

Skalaen er 0,5 til 5 i halve steg. Ingen verdier ligger utenfor skalaen, og det finnes ingen 0-ratinger.

### 3. Tidsstempel

`timestamp` er Unix-tid (sekunder siden 1970). Vi konverterer til datetime for å se hvilken periode dataene dekker.

In [ ]:
for df in [df_ratings_small, df_ratings]:
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s")

print("ratings_small:", df_ratings_small["timestamp"].min(), "til", df_ratings_small["timestamp"].max())
print("ratings:      ", df_ratings["timestamp"].min(), "til", df_ratings["timestamp"].max())

### 4. Fordeling av ratings

In [ ]:
df_ratings["rating"].describe()

In [ ]:
df_ratings["rating"].value_counts().sort_index().plot(kind="bar")
plt.title("Fordeling av ratings (ratings.csv)")
plt.xlabel("Rating")
plt.ylabel("Antall ratings")
plt.show()

Ratingene er venstreskjeve: 4 er vanligst, og hele tall brukes oftere enn halve. Gjennomsnittet er ca. 3,5.

### 5. Antall ratings per bruker og per film

In [ ]:
per_bruker = df_ratings.groupby("userId").size()
per_film = df_ratings.groupby("movieId").size()

print("Antall brukere:", per_bruker.shape[0])
print("Antall filmer: ", per_film.shape[0])
pd.DataFrame({"per bruker": per_bruker.describe(), "per film": per_film.describe()})

In [ ]:
per_film.hist(bins=50, log=True)
plt.title("Antall ratings per film (log-skala på y-aksen)")
plt.xlabel("Antall ratings")
plt.ylabel("Antall filmer")
plt.show()

Begge fordelingene er svært skjeve. Medianfilmen har bare 8 ratings, mens de mest populære har over 90 000. En gjennomsnittsrating for en film med få stemmer er lite pålitelig, så i analysene bør vi sette en nedre grense for antall ratings.

### 6. ratings_small vs. ratings

In [ ]:
print("ratings_small: brukere =", df_ratings_small["userId"].nunique(), ", filmer =", df_ratings_small["movieId"].nunique())
print("ratings:       brukere =", df_ratings["userId"].nunique(), ", filmer =", df_ratings["movieId"].nunique())

felles = df_ratings_small.merge(df_ratings, on=["userId", "movieId"])
print("Par (userId, movieId) fra ratings_small som også finnes i ratings:", len(felles), "av", len(df_ratings_small))

ratings_small er **ikke** et utsnitt av ratings. Bare en liten andel av parene går igjen, så `userId` betyr trolig ikke samme person i de to filene. De to filene må derfor ikke blandes.

## Kobling mellom datasettene ##

ratings → links via `movieId`, og links → movies_metadata via `tmdbId` = `id`. Vi sjekker hvor mye som faktisk lar seg koble.

In [ ]:
print("movieId i ratings som finnes i links:            ", df_ratings["movieId"].isin(df_links["movieId"]).mean())
print("movieId i ratings_small som finnes i links_small:", df_ratings_small["movieId"].isin(df_links_small["movieId"]).mean())
print("movieId i ratings_small som finnes i links:      ", df_ratings_small["movieId"].isin(df_links["movieId"]).mean())

In [ ]:
# Leser bare id-kolonnen fra movies_metadata. Feilformaterte id-er (datoer) blir NaN.
metadata_id = pd.to_numeric(pd.read_csv("data/movies_metadata.csv", usecols=["id"])["id"], errors="coerce")

tmdb = df_links["tmdbId"].dropna()
print("tmdbId i links som finnes i movies_metadata:", tmdb.isin(metadata_id).mean())
print("Antall som ikke finnes:", (~tmdb.isin(metadata_id)).sum())

Koblingen er nesten fullstendig. Alle `movieId` i ratings finnes i links, men 46 ratings i ratings_small peker på filmer som ikke finnes i links (de 10 filmene fra seksjon 3 over). Av `tmdbId` i links finnes 161 ikke i movies_metadata. Ratings på disse filmene kan ikke knyttes til metadata, så de mister tittel, sjanger osv. ved innsetting.